In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e8/sample_submission.csv
/kaggle/input/playground-series-s5e8/train.csv
/kaggle/input/playground-series-s5e8/test.csv


# Data Reading

In [2]:
train = pd.read_csv('/kaggle/input/playground-series-s5e8/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e8/test.csv')

In [3]:
# checking for missing values
print(train.isna().sum())
print()
print(test.isna().sum())

id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
dtype: int64


In [4]:
# statistical description of the dataset
# for training
train.describe()

,id,age,balance,day,duration,campaign,pdays,previous,y
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,374999.500000,40.926395,1204.067397,16.117209,256.229144,2.577008,22.412733,0.298545,0.120651
std,216506.495284,10.098829,2836.096759,8.250832,272.555662,2.718514,77.319998,1.335926,0.325721
min,0.000000,18.000000,-8019.000000,1.000000,1.000000,1.000000,-1.000000,0.000000,0.000000
25%,187499.750000,33.000000,0.000000,9.000000,91.000000,1.000000,-1.000000,0.000000,0.000000
50%,374999.500000,39.000000,634.000000,17.000000,133.000000,2.000000,-1.000000,0.000000,0.000000
75%,562499.250000,48.000000,1390.000000,21.000000,361.000000,3.000000,-1.000000,0.000000,0.000000
max,749999.000000,95.000000,99717.000000,31.000000,4918.000000,63.000000,871.000000,200.000000,1.000000


In [5]:
# for testing
test.describe()

,id,age,balance,day,duration,campaign,pdays,previous
count,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000
mean,874999.500000,40.932332,1197.426352,16.116068,255.342260,2.573548,22.280028,0.303728
std,72168.927986,10.081613,2741.520699,8.258509,271.404326,2.709661,76.915879,1.384574
min,750000.000000,18.000000,-8019.000000,1.000000,3.000000,1.000000,-1.000000,0.000000
25%,812499.750000,33.000000,0.000000,9.000000,91.000000,1.000000,-1.000000,0.000000
50%,874999.500000,39.000000,631.000000,17.000000,133.000000,2.000000,-1.000000,0.000000
75%,937499.250000,48.000000,1389.000000,21.000000,353.000000,3.000000,-1.000000,0.000000
max,999999.000000,95.000000,98517.000000,31.000000,4918.000000,58.000000,871.000000,150.000000


In [6]:
# printing the columns of 'object' datatype
object_cols = train.select_dtypes(include="object").columns.tolist()
print(f"The object columns are: \n{object_cols}")

The object columns are: 
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [7]:
# printing the unique values of each object columns
for col_name in object_cols:
    print(f"{col_name}:\n{sorted(train[col_name].unique())}\n")

job:
['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']

marital:
['divorced', 'married', 'single']

education:
['primary', 'secondary', 'tertiary', 'unknown']

default:
['no', 'yes']

housing:
['no', 'yes']

loan:
['no', 'yes']

contact:
['cellular', 'telephone', 'unknown']

month:
['apr', 'aug', 'dec', 'feb', 'jan', 'jul', 'jun', 'mar', 'may', 'nov', 'oct', 'sep']

poutcome:
['failure', 'other', 'success', 'unknown']



# Data Processing

In [8]:
# importing necessary modules
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [9]:
# partitioning into feature matrix X and target vector y 
X = train.drop(['id', 'y'], axis=1)
y = train['y']

# dropping id from test dataset
test.drop(['id'], axis=1, inplace=True)

In [10]:
# applying columntransformer to apply standard scaler to numerical values and one-hot encoding to
# categorical values
data_processor = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(), X.select_dtypes(include=['float64', 'int64']).columns),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), object_cols)
    ]
)
data_processor

ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                 Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome'])])

# Model Training

In [11]:
# splitting the dataset for training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
# obtained this hyperparameters from Optuna
best_params={
    'max_leaves': 71,
    'min_child_weight': 7.627525480669875,
    'learning_rate': 0.07600597724564652,
    'subsample': 0.9990411964287336,
    'colsample_bylevel': 0.962210548867428,
    'colsample_bytree': 0.677377783758824,
    'reg_alpha': 4.02840911563676,
    'reg_lambda': 0.6770091573319115,
    'n_estimators': 1401
}

# Fit data processor separately
processor_fitted = data_processor.fit(X_train)
# Transform train and val for XGBoost
X_train_proc = processor_fitted.transform(X_train)
X_val_proc = processor_fitted.transform(X_val)

model = XGBClassifier(
    **best_params,
    objective='binary:logistic',
    tree_method='hist',
    device="cuda",
    eval_metric="auc",
    random_state=42
)
model

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.962210548867428, colsample_bynode=None,
              colsample_bytree=0.677377783758824, device='cuda',
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.07600597724564652,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=71,
              min_child_weight=7.627525480669875, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1401,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [13]:
# creating the pipeline to apply the transformations
pipeline = Pipeline(
    steps=[
        ('data_processor', data_processor),
        ('xgb_classifier', model)
    ])
pipeline

Pipeline(steps=[('data_processor',
                 ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                                  Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome'])])),
                ('xgb_classifier',
                 XGBC...
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.07600597724564652, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=None,
                               max_leaves=71,
                               min_child_weight=7.627525480669875, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=1401, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

In [14]:
%%time
# fitting the pipeline which already contains the proecssing steps and model
pipeline.named_steps['xgb_classifier'].fit(
    X_train_proc, y_train,
    eval_set=[(X_val_proc, y_val)],
    early_stopping_rounds=250,
    verbose=False
)

/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


CPU times: user 14.4 s, sys: 480 ms, total: 14.9 s
Wall time: 12.4 s


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.962210548867428, colsample_bynode=None,
              colsample_bytree=0.677377783758824, device='cuda',
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.07600597724564652,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=71,
              min_child_weight=7.627525480669875, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1401,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [15]:
# calculate the accuracy, roc-auc
y_pred = pipeline.predict(X_val)
print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_val, y_pred)}")
print(f"Classification Report: \n{classification_report(y_val, y_pred)}")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [14:27:23] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.9364
ROC AUC Score: 0.8254448781120963
Classification Report: 
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    131902
           1       0.77      0.68      0.72     18098

    accuracy                           0.94    150000
   macro avg       0.86      0.83      0.84    150000
weighted avg       0.93      0.94      0.93    150000



# Submission

In [16]:
sub = pd.read_csv('/kaggle/input/playground-series-s5e8/sample_submission.csv')
test_preds = pipeline.predict_proba(test)
test_preds_proba = test_preds[:, 1]
submission = pd.DataFrame({
    'id': sub['id'],
    'y': test_preds_proba
})
submission.to_csv('submission.csv', index=False)